**Objectives**:
<ul>
<li>Define structured output formats using Pydantic models</li>
<li>Enhance an existing agent class to support structured outputs</li>
<li>Validate and parse responses to ensure they meet defined formats</li>

In [11]:
!pip install langchain openai langchain_openai
from typing import List, Any, Annotated, Optional, Type
from pydantic import BaseModel, Field
import json
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, BaseMessage
from langchain_core.output_parsers import JsonOutputParser

In [15]:
class ActionItem(BaseModel):
  task: Annotated[str, Field(description="The task to be completed")]
  assignee: Annotated[str, Field(description="Person responsible for the task")]
  due_date: Annotated[str, Field(description="When the task should be completed")]

In [16]:
class MeetingSummary(BaseModel):
  title:Annotated[str, Field(description="The title of the meeting")]
  date:Annotated[str, Field(description="The date of the meeting")]
  participants:Annotated[List[str], Field(description="The participants of the meeting")]
  key_points:Annotated[List[str], Field(description="The key points of the meeting")]
  action_items:Annotated[List[ActionItem], Field(description="The action items for the meeting")]


In [ ]:
class StructuredAgent:
    def __init__(
        self,
        role: str = "Personal Assistant",
        instructions: str = "Help users with any question",
        model: str = "gpt-4o-mini",
        temperature: float = 0.0,
        tools: List[Any] = None,
        output_schema: Optional[Type[BaseModel]] = None # Added output_schema parameter
    ):

        self.model = model
        self.role = role
        self.instructions = instructions
        self.tools = tools or []
        self.output_schema = output_schema # Initialize output_schema

        self.llm = ChatOpenAI(
            model=model,
            api_key="",
            temperature=temperature
        )

    def invoke(self, user_message: str) -> dict:
        messages = [SystemMessage(content=f"You're an AI Agent and your role is {self.role}. Your instructions: {self.instructions}")]
        messages.append(HumanMessage(content=user_message))
        if self.output_schema:
            # Use with_structured_output for direct Pydantic model output
            structured_llm = self.llm.with_structured_output(self.output_schema)
            ai_message = structured_llm.invoke(messages)
            return ai_message.dict() # Convert Pydantic object to dict
        else:
            ai_message = self.llm.invoke(messages)
            return {"response": ai_message.content}

In [20]:
meeting_agent = StructuredAgent(
    role="Meeting Assistant",
    instructions="Summarize meetings and track action items in a structured format",
    output_schema=MeetingSummary # Pass the MeetingSummary Pydantic model as output_schema
)

meeting_transcript = """
Project Planning Meeting - March 15, 2024
Attendees: John, Sarah, Mike
Discussion:

* Reviewed Q1 project timeline
* Discussed resource allocation
* Identified potential risks
Next steps:
* John will update the project plan by next Friday
* Sarah needs to coordinate with the design team by Wednesday
* Mike will prepare the risk assessment document by end of month
"""

summary = meeting_agent.invoke(meeting_transcript)
print(json.dumps(summary, indent=2))
validated_summary = MeetingSummary(**summary)
print("Meeting Title:", validated_summary.title)
print("\nParticipants:")
for participant in validated_summary.participants:
    print(f"- {participant}")
print("\nAction Items:")
for item in validated_summary.action_items:
    print(f"- {item.task} (Assigned to: {item.assignee}, Due: {item.due_date})")

{
  "title": "Project Planning Meeting",
  "date": "March 15, 2024",
  "participants": [
    "John",
    "Sarah",
    "Mike"
  ],
  "key_points": [
    "Reviewed Q1 project timeline",
    "Discussed resource allocation",
    "Identified potential risks"
  ],
  "action_items": [
    {
      "task": "Update the project plan",
      "assignee": "John",
      "due_date": "March 22, 2024"
    },
    {
      "task": "Coordinate with the design team",
      "assignee": "Sarah",
      "due_date": "March 20, 2024"
    },
    {
      "task": "Prepare the risk assessment document",
      "assignee": "Mike",
      "due_date": "March 31, 2024"
    }
  ]
}
Meeting Title: Project Planning Meeting

Participants:
- John
- Sarah
- Mike

Action Items:
- Update the project plan (Assigned to: John, Due: March 22, 2024)
- Coordinate with the design team (Assigned to: Sarah, Due: March 20, 2024)
- Prepare the risk assessment document (Assigned to: Mike, Due: March 31, 2024)


/tmp/ipykernel_5863/634071967.py:31: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  return ai_message.dict() # Convert Pydantic object to dict
